# HarrisWGAN demo

In [1]:
import argparse
import os
import sys
from datetime import datetime as dt

import numpy as np
import xarray as xr

sys.path.append("/p/home/jusers/lehner3/hdfml/shared/downscaling_benchmark/models")
sys.path.append("/p/home/jusers/lehner3/hdfml/shared/downscaling_benchmark/utils")
from harris_wgan_model import DiscriminatorHarris, GeneratorHarris, HarrisWGAN

# from handle_data_unet import HandleUnetData

In [2]:
import gc
import glob
import os
from typing import List, Tuple

import numpy as np
import tensorflow as tf
import xarray as xr


def split_in_tar(
    ds: xr.Dataset,
    predictands: List = None,
    predictors: List = None,
    static_vars: List = None,
) -> Tuple[xr.Dataset, xr.Dataset]:
    """
    Split data array with variables-dimension into input and target data for downscaling
    :param da: The unsplitted data array
    :param target_var: Name of target variable which should consttute the first channel
    :param predictands: List of selected predictand variables; parse None to use
                        all predictands (vars with suffix _tar)
    :param predictors: List of selected predictor variables; parse None to use all predictors (vars with suffix _in)
    :return: The split data array.
    """
    varnames = list(ds.data_vars)

    if predictors is None:
        invars = [var for var in varnames if var.endswith("_in")]
    else:
        assert all(
            [predictor in varnames for predictor in predictors]
        ), f"At least one predictor is not a data variable. Available variables are {*varnames,}"
        invars = list(predictors)
    if predictands is None:
        tarvars = [var for var in varnames if var.endswith("_tar")]
    else:
        assert all(
            [predictand in varnames for predictand in predictands]
        ), f"At least one predictor is not a data variable. Available variables are {*varnames,}"
        tarvars = list(predictands)

    if static_vars is None:
        ds_in, ds_tar = ds[invars], ds[tarvars]

        return ds_in, ds_tar
    else:
        assert all(
            [static_var in varnames for static_var in static_vars]
        ), f"At least ostatic high-res is not a data variable. Available variables are {*varnames,}"
        statvars = list(static_vars)

        ds_in, ds_tar, ds_stat = ds[invars], ds[tarvars], ds[statvars]

        return ds_in, ds_tar, ds_stat


def reshape_ds(ds):
    """
    Convert a xarray dataset to a data-array where the variables will constitute the last dimension (channel last)
    :param ds: the xarray dataset with dimensions (dims)
    :return da: the data-array with dimensions (dims, variables)
    """
    da = ds.to_array(dim="variables")
    da = da.transpose(..., "variables")
    return da


def make_tf_dataset_allmem(
    ds: xr.Dataset,
    batch_size: int,
    predictands: List,
    predictors: List,
    static_vars: List,
    lshuffle: bool = True,
    shuffle_samples: int = 20000,
    named_targets: bool = False,
    var_tar2in: str = None,
    lrepeat: bool = True,
    drop_remainder: bool = True,
) -> tf.data.Dataset:
    """
    Build-up TensorFlow dataset from a generator based on the xarray-data array.
    NOTE: All data is loaded into memory
    :param ds: the xarray dataset. Input variable names must carry the suffix '_in', whereas it must be '_tar' for target variables
    :param batch_size: number of samples per mini-batch
    :param predictands: List of selected predictand variables
    :param predictors: List of selected predictor variables; parse None to use all predictors (vars with suffix _in)
    :param lshuffle: flag if shuffling should be applied to dataset
    :param shuffle_samples: number of samples to load before applying shuffling
    :param named_targets: flag if target of TF dataset should be dictionary with named target variables
    :param var_tar2in: name of target variable to be added to input (used e.g. for adding high-resolved topography
                                                                        to the input)
    :param lrepeat: flag if dataset should be repeated
    :param drop_remainder: flag if samples will be dropped in case batch size is not a divisor of # data samples
    :param with_horovod: flag to trigger horovod-based distributed dataset creation
    :param lembed: flag to trigger temporal embedding (not implemented yet!)
    """

    # add time dimension to constant variables
    for var in ds.data_vars:
        if "time" not in ds[var].dims:
            ds[var] = ds[var].expand_dims({"time": ds["time"]}, axis=0)

    ds_in, ds_tar, ds_stat = split_in_tar(
        ds, predictands=predictands, predictors=predictors, static_vars=static_vars
    )

    # convert dataset to data arrays and load into memory
    da_in, da_tar, da_stat = (
        reshape_ds(ds_in).astype("float32", copy=True),
        reshape_ds(ds_tar).astype("float32", copy=True),
        reshape_ds(ds_stat).astype("float32", copy=True),
    )

    if var_tar2in is not None:
        # NOTE: * The order of the following operation must be the same as in StreamMonthlyNetCDF.getitems
        #       * The following operation order must concatenate var_tar2in by da_in to ensure
        #         that the variable appears at first place. This is required to avoid
        #         that var_tar2in becomes a predeictand when slicing takes place in tf_split
        da_in = xr.concat([da_tar.sel({"variables": var_tar2in}), da_in], "variables")

    varnames_tar = da_tar["variables"].values

    def gen_named(darr_in, darr_tar):
        # darr_in, darr_tar = darr_in.load(), darr_tar.load()
        ntimes = len(darr_in["time"])
        for t in range(ntimes):
            tar_now = darr_tar.isel({"time": t})
            yield tuple(
                (
                    darr_in.isel({"time": t}).values,
                    {
                        var: tar_now.sel({"variables": var}).values
                        for var in varnames_tar
                    },
                )
            )

    def gen_unnamed(darr_in, darr_tar):
        # darr_in, darr_tar = darr_in.load(), darr_tar.load()
        ntimes = len(darr_in["time"])
        for t in range(ntimes):
            yield tuple(
                (darr_in.isel({"time": t}).values, darr_tar.isel({"time": t}).values)
            )

    def gen_dict(darr_in, darr_tar, darr_stat):
        ntimes = len(darr_in["time"])
        for t in range(ntimes):
            yield tuple(
                (
                    {
                        "lo_res_inputs": darr_in.isel({"time": t}).values,
                        "hi_res_inputs": darr_stat.isel({"time": t}).values,
                    },
                    {"output": darr_tar.isel({"time": t}).values},
                )
            )

    if named_targets is True:
        gen_now = gen_named
    elif static_vars is not None:
        gen_now = gen_dict
    else:
        gen_now = gen_unnamed

    # create output signatures from first sample
    if static_vars is None:
        s0 = next(iter(gen_now(da_in, da_tar)))
        sample_spec_in = tf.TensorSpec(s0[0].shape, dtype=s0[0].dtype)
        if named_targets is True:
            sample_spec_tar = {
                var: tf.TensorSpec(s0[1][var].shape, dtype=s0[1][var].dtype)
                for var in varnames_tar
            }
        else:
            sample_spec_tar = tf.TensorSpec(s0[1].shape, dtype=s0[1].dtype)

        # re-instantiate the generator and build TF dataset
        gen_train = gen_now(da_in, da_tar)

    else:
        s0 = next(iter(gen_now(da_in, da_tar, da_stat)))

        sample_spec_in = {
            "lo_res_inputs": tf.TensorSpec(
                s0[0]["lo_res_inputs"].shape, dtype=s0[0]["lo_res_inputs"].dtype
            ),
            "hi_res_inputs": tf.TensorSpec(
                s0[0]["hi_res_inputs"].shape, dtype=s0[0]["hi_res_inputs"].dtype
            ),
        }

        sample_spec_tar = {
            "output": tf.TensorSpec(s0[1]["output"].shape, dtype=s0[1]["output"].dtype)
        }

        # re-instantiate the generator and build TF dataset
        gen_train = gen_now(da_in, da_tar, da_stat)

    data_iter = tf.data.Dataset.from_generator(
        lambda: gen_train, output_signature=(sample_spec_in, sample_spec_tar)
    )

    # Notes:
    # * cache is reuqired to make repeat work properly on datasets based on generators
    #   (see https://stackoverflow.com/questions/60226022/tf-data-generator-keras-repeat-does-not-work-why)
    # * repeat must be applied after shuffle to get varying mini-batches per epoch
    # * batch-size is increased to allow substepping in train_step
    if lshuffle > 1:
        data_iter = (
            data_iter.cache()
            .shuffle(shuffle_samples)
            .batch(batch_size, drop_remainder=drop_remainder)
        )
    else:
        data_iter = data_iter.cache().batch(batch_size, drop_remainder=drop_remainder)

    if lrepeat:
        data_iter = data_iter.repeat()

    # clean-up to free some memory
    # free_mem([da, da_in, da_tar, varnames_tar])
    del ds
    del ds_in
    del ds_tar
    del da_in
    del da_tar
    gc.collect()

    return data_iter

In [3]:
# set diretcories and (hyper-)parameters for WGAN
t2m_test_file = "/p/scratch/deepacf/maelstrom/maelstrom_data/ap5/downscaling_benchmark_dataset/benchmark_t2m/dataset/testdata/downscaling_benchmark_t2m_allmem_test.nc"
# datadir = "/p/scratch/deepacf/maelstrom/maelstrom_data/ap5_michael/preprocessed_era5_ifs/netcdf_data/all_files/"
# outdir = "/p/project/deepacf/maelstrom/langguth1/downscaling_jsc_repo/downscaling_unet/trained_models"

lr_gen = 5.0e-05
lr_gen_end = lr_gen / 10.0
lr_critic = 1.0e-06
lr_decay = True
nepochs = 1
d_steps = 6
batch_size_demo = 2

# Read training and validation data
# ds_train, ds_val = xr.open_dataset(os.path.join(datadir, "era5_to_ifs_train_corrected.nc")), \
#                   xr.open_dataset(os.path.join(datadir, "era5_to_ifs_val_corrected.nc"))

ds_train = xr.open_dataset(t2m_test_file)
ds_val = ds_train
z_branch = False
print("Datasets for trining, validation and testing loaded.")

# wgan_model = HarrisWGAN(GeneratorHarris, DiscriminatorHarris,
#                  {"lr_decay": lr_decay, "lr_gen": lr_gen, "lr_critic": lr_critic, "lr_gen_end": lr_gen_end,
#                   "train_epochs": nepochs, "d_steps": d_steps, "z_branch": z_branch})

Datasets for trining, validation and testing loaded.


In [4]:
tfds = make_tf_dataset_allmem(
    ds_train,
    batch_size_demo * 7,
    ["t_2m_tar"],
    ["t2m_in", "sp_in", "sshf_in", "t115_in"],
    ["fr_land_tar", "hsurf_tar"],
)
tfds_val = make_tf_dataset_allmem(
    ds_train,
    batch_size_demo * 7,
    ["t_2m_tar"],
    ["t2m_in", "sp_in", "sshf_in", "t115_in"],
    ["fr_land_tar", "hsurf_tar"],
)

2024-07-12 21:21:34.016244: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38364 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:03:00.0, compute capability: 8.0
2024-07-12 21:21:34.019132: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 38364 MB memory:  -> device: 1, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:44:00.0, compute capability: 8.0
2024-07-12 21:21:34.022471: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 38364 MB memory:  -> device: 2, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:84:00.0, compute capability: 8.0
2024-07-12 21:21:34.025042: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 38364 MB memory:  -> device: 3, name: NVIDIA A100-SXM4-40GB, pci bu

In [5]:
tfds, tfds_val

(<RepeatDataset element_spec=({'lo_res_inputs': TensorSpec(shape=(14, 32, 36, 4), dtype=tf.float32, name=None), 'hi_res_inputs': TensorSpec(shape=(14, 128, 144, 2), dtype=tf.float32, name=None)}, {'output': TensorSpec(shape=(14, 128, 144, 1), dtype=tf.float32, name=None)})>,
 <RepeatDataset element_spec=({'lo_res_inputs': TensorSpec(shape=(14, 32, 36, 4), dtype=tf.float32, name=None), 'hi_res_inputs': TensorSpec(shape=(14, 128, 144, 2), dtype=tf.float32, name=None)}, {'output': TensorSpec(shape=(14, 128, 144, 1), dtype=tf.float32, name=None)})>)

In [6]:
import importlib

from model_engine import ModelEngine

In [27]:
importlib.reload(sys.modules["model_engine"])
importlib.reload(sys.modules["harris_wgan_model"])
# steps to run
shape_in = {
    "harris_generator": {
        "lo_res_inputs": (32, 36, 4),
        "hi_res_inputs": (128, 144, 2),
        "noise_input": (32, 36, 4),
    },
    "harris_discriminator": {
        "lo_res_inputs": (32, 36, 4),
        "hi_res_inputs": (128, 144, 2),
        "output": (128, 144, 1),
    },
}
varnames_tar = "t_2m_tar"
hparams_dict = dict()
model_savedir = ""
steps_per_epoch = 100

model_instance = ModelEngine("harris_wgan")
# data prep here
model = model_instance(
    shape_in, list(varnames_tar), hparams_dict, model_savedir, "demo"
)
model.compile(**model.compile_options)
history = model.fit(
    x=tfds,
    epochs=model.hparams["nepochs"],
    steps_per_epoch=steps_per_epoch,
    validation_data=tfds_val,
    validation_steps=300,
    verbose=2,
    **model.fit_options
)

generator_input shape: (None, 32, 36, 4)
constants_input shape: (None, 128, 144, 2)
upscaled constants shape: (None, 32, 36, 128)
noise_input shape: (None, 32, 36, 4)
Shape after first concatenate: (None, 32, 36, 136)
End of first residual block
Shape after first residual block: (None, 32, 36, 128)
Shape after upsampling step 1: (None, 128, 144, 128)
Shape after residual block: (None, 128, 144, 128)
Shape after second concatenate: (None, 128, 144, 130)
Shape after third residual block: (None, 128, 144, 128)
Output shape: (None, 128, 144, 1)
generator_input shape: (None, 32, 36, 4)
constants_input shape: (None, 128, 144, 2)
generator_output shape: (None, 128, 144, 1)
upscaled constants shape: (None, 32, 36, 512)
Shape after lo-res concatenate: (None, 32, 36, 516)
Shape after hi-res concatenate: (None, 128, 144, 3)
Shape of lo-res input after residual block: (None, 32, 36, 1024)
Shape of hi_res_input after upsampling step 1: (None, 32, 36, 1024)
Shape of hi-res input after residual block

ValueError: in user code:

    File "/p/software/juwelsbooster/stages/2023/software/TensorFlow/2.11.0-foss-2022a-CUDA-11.7/lib/python3.10/site-packages/keras/engine/training.py", line 1249, in train_function  *
        return step_function(self, iterator)
    File "/p/software/juwelsbooster/stages/2023/software/TensorFlow/2.11.0-foss-2022a-CUDA-11.7/lib/python3.10/site-packages/keras/engine/training.py", line 1233, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "/p/software/juwelsbooster/stages/2023/software/TensorFlow/2.11.0-foss-2022a-CUDA-11.7/lib/python3.10/site-packages/keras/engine/training.py", line 1222, in run_step  **
        outputs = model.train_step(data)
    File "/p/home/jusers/lehner3/hdfml/shared/downscaling_benchmark/models/harris_wgan_model.py", line 361, in train_step
        gp = self.gradient_penalty(sample_iter, gen_out)
    File "/p/home/jusers/lehner3/hdfml/shared/downscaling_benchmark/models/harris_wgan_mo

In [ ]:
model_name = "harriswgan_lr1e-05_epochs1_opt_split_era5_ifs"

savedir = os.path.join("../downscaling_harriswgan/trained_models/", model_name)
os.makedirs(savedir, exist_ok=True)

In [ ]:
model.generator.save(
    os.path.join(savedir, "harriswgan_lr1e-05_epochs30_demo_generator")
)
model.critic.save(
    os.path.join(savedir, "harriswgan_lr1e-05_epochs30_demo_discriminator")
)